# 第十五章：深度殘差網路中的恆等映射

## Identity Mappings in Deep Residual Networks
## He, Zhang, Ren, Sun (ECCV 2016)

這篇論文改進了原始 ResNet，提出「預活化」殘差塊，創造更乾淨的恆等映射路徑。

**核心貢獻**：
- 預活化架構（Pre-activation）
- 更好的梯度流動
- 成功訓練 1001 層網路

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# 設定隨機種子
torch.manual_seed(42)
np.random.seed(42)

# 檢查裝置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用裝置：{device}")

## 1. 問題：原始 ResNet 的恆等路徑被阻斷

原始 ResNet 塊：
```
x → Conv → BN → ReLU → Conv → BN → (+x) → ReLU → y
```

問題：最後的 ReLU 阻斷了恆等映射！

$$y = \text{ReLU}(F(x) + x)$$

當 $x < 0$ 時，恆等映射被截斷。

In [ ]:
class OriginalBasicBlock(nn.Module):
    """
    原始 ResNet 基本塊（後置活化）
    
    x → Conv → BN → ReLU → Conv → BN → (+x) → ReLU
    """
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 捷徑連接（如果維度不匹配）
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        # 殘差路徑
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        # 加上捷徑
        out = out + self.shortcut(x)
        
        # 後置 ReLU（阻斷恆等路徑！）
        out = F.relu(out)
        
        return out

# 測試原始塊
original_block = OriginalBasicBlock(64, 64)
x = torch.randn(1, 64, 32, 32)
y = original_block(x)

print("原始 ResNet 塊：")
print(f"  輸入形狀: {x.shape}")
print(f"  輸出形狀: {y.shape}")
print(f"  最後有 ReLU：是（阻斷恆等路徑）")

## 2. 解決方案：預活化殘差塊

預活化 ResNet 塊：
```
x → BN → ReLU → Conv → BN → ReLU → Conv → (+x) → y
```

數學表示：
$$y = F'(x) + x$$

恆等路徑完全乾淨！

In [ ]:
class PreActBasicBlock(nn.Module):
    """
    預活化 ResNet 基本塊
    
    x → BN → ReLU → Conv → BN → ReLU → Conv → (+x)
    """
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        # 預活化順序：BN → ReLU → Conv
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        
        # 捷徑連接
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 1, stride, bias=False)
    
    def forward(self, x):
        # 預活化
        out = F.relu(self.bn1(x))
        
        # 捷徑路徑（從預活化後的特徵取）
        shortcut = self.shortcut(out)
        
        # 殘差路徑
        out = self.conv1(out)
        out = F.relu(self.bn2(out))
        out = self.conv2(out)
        
        # 加上捷徑（無後置 ReLU！）
        out = out + shortcut
        
        return out

# 測試預活化塊
preact_block = PreActBasicBlock(64, 64)
y_preact = preact_block(x)

print("預活化 ResNet 塊：")
print(f"  輸入形狀: {x.shape}")
print(f"  輸出形狀: {y_preact.shape}")
print(f"  最後有 ReLU：否（恆等路徑乾淨）")

In [ ]:
# 視覺化兩種架構的差異
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

def draw_block_diagram(ax, title, is_preact=False):
    """繪製殘差塊的架構圖"""
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 14)
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    # 顏色定義
    colors = {
        'Conv': '#2196F3',
        'BN': '#4CAF50',
        'ReLU': '#FF9800',
        'Add': '#9C27B0'
    }
    
    # 恆等路徑
    ax.annotate('', xy=(2, 1.5), xytext=(2, 12.5),
                arrowprops=dict(arrowstyle='->', color='blue', lw=3))
    ax.text(0.5, 7, 'Identity\nPath', fontsize=10, ha='center', color='blue')
    
    # 輸入
    ax.text(5, 13.5, 'Input x', fontsize=12, ha='center', fontweight='bold')
    ax.annotate('', xy=(5, 12.5), xytext=(5, 13),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    if is_preact:
        # 預活化順序
        operations = ['BN', 'ReLU', 'Conv', 'BN', 'ReLU', 'Conv']
        y_positions = [12, 10.5, 9, 7.5, 6, 4.5]
    else:
        # 原始順序
        operations = ['Conv', 'BN', 'ReLU', 'Conv', 'BN']
        y_positions = [12, 10.5, 9, 7.5, 6]
    
    # 繪製操作
    for op, y in zip(operations, y_positions):
        rect = plt.Rectangle((4, y-0.4), 2, 0.8, 
                             facecolor=colors[op], edgecolor='black', lw=2)
        ax.add_patch(rect)
        ax.text(5, y, op, fontsize=11, ha='center', va='center', 
               fontweight='bold', color='white')
        
        # 箭頭到下一個
        if op != operations[-1]:
            idx = operations.index(op)
            ax.annotate('', xy=(5, y_positions[idx+1]+0.5), xytext=(5, y-0.5),
                        arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    
    # 加法節點
    add_y = 2.5
    circle = plt.Circle((5, add_y), 0.5, facecolor='white', edgecolor='purple', lw=3)
    ax.add_patch(circle)
    ax.text(5, add_y, '+', fontsize=20, ha='center', va='center', fontweight='bold')
    
    # 連接到加法
    ax.plot([2, 5], [1.5, add_y-0.5], 'b-', lw=2)
    last_y = y_positions[-1]
    ax.annotate('', xy=(5, add_y+0.6), xytext=(5, last_y-0.5),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    
    if not is_preact:
        # 原始：加法後有 ReLU
        relu_y = 1
        rect = plt.Rectangle((4, relu_y-0.4), 2, 0.8, 
                             facecolor='#F44336', edgecolor='black', lw=2)
        ax.add_patch(rect)
        ax.text(5, relu_y, 'ReLU', fontsize=11, ha='center', va='center', 
               fontweight='bold', color='white')
        ax.annotate('', xy=(5, relu_y+0.5), xytext=(5, add_y-0.6),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
        
        # 警告
        ax.text(8, 1, 'ReLU blocks\nidentity!', fontsize=10, color='red',
               bbox=dict(boxstyle='round', facecolor='#FFCDD2', alpha=0.8))
        output_y = 0
    else:
        # 預活化：無後置 ReLU
        ax.text(8, 2.5, 'Clean\nidentity!', fontsize=10, color='green',
               bbox=dict(boxstyle='round', facecolor='#C8E6C9', alpha=0.8))
        output_y = 1.5
    
    # 輸出
    ax.annotate('', xy=(5, output_y), xytext=(5, add_y-0.6 if is_preact else relu_y-0.5),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.text(5, output_y-0.5, 'Output y', fontsize=12, ha='center', fontweight='bold')

draw_block_diagram(axes[0], 'Original ResNet Block (Post-activation)', is_preact=False)
draw_block_diagram(axes[1], 'Pre-activation ResNet Block', is_preact=True)

plt.tight_layout()
plt.savefig('block_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("圖片已儲存：block_comparison.png")

## 3. 恆等映射測試

當殘差路徑學習「什麼都不做」時，輸出應該等於輸入。

測試：將殘差路徑的權重設為零，比較輸出和輸入的差異。

In [ ]:
def test_identity_mapping(block, num_tests=100):
    """
    測試恆等映射的純淨度
    
    當殘差路徑為零時，輸出應該等於輸入
    """
    block.eval()
    
    # 將殘差路徑的卷積權重設為零
    with torch.no_grad():
        for name, param in block.named_parameters():
            if 'conv' in name and 'weight' in name:
                param.zero_()
    
    errors = []
    for _ in range(num_tests):
        x = torch.randn(1, 64, 8, 8)
        with torch.no_grad():
            y = block(x)
        
        # 計算輸出和輸入的差異
        error = (y - x).abs().mean().item()
        errors.append(error)
    
    return np.mean(errors), np.std(errors)

# 測試兩種塊
original_block_test = OriginalBasicBlock(64, 64)
preact_block_test = PreActBasicBlock(64, 64)

error_original, std_original = test_identity_mapping(original_block_test)
error_preact, std_preact = test_identity_mapping(preact_block_test)

print("恆等映射測試（殘差路徑 = 0）：")
print("="*50)
print(f"原始 ResNet 誤差: {error_original:.6f} ± {std_original:.6f}")
print(f"預活化 ResNet 誤差: {error_preact:.6f} ± {std_preact:.6f}")
print("="*50)
print(f"\n結論：預活化的恆等映射更{'純淨' if error_preact < error_original else '差'}")
print("（誤差越小 = 恆等路徑越乾淨）")

In [ ]:
# 視覺化恆等映射測試
fig, ax = plt.subplots(figsize=(10, 6))

blocks = ['Original ResNet\n(Post-activation)', 'Pre-activation ResNet']
errors = [error_original, error_preact]
stds = [std_original, std_preact]
colors = ['#F44336', '#4CAF50']

bars = ax.bar(blocks, errors, yerr=stds, color=colors, edgecolor='black', 
              capsize=10, alpha=0.8)

ax.set_ylabel('Mean Absolute Error', fontsize=12)
ax.set_title('Identity Mapping Test\n(Lower = Cleaner Identity Path)', fontsize=14)
ax.set_ylim(0, max(errors) * 1.5)

# 標註數值
for bar, err in zip(bars, errors):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{err:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('identity_test.png', dpi=150, bbox_inches='tight')
plt.show()

print("圖片已儲存：identity_test.png")

## 4. 梯度流動分析

預活化架構的梯度流動更順暢。

In [ ]:
class SimpleResNet(nn.Module):
    """
    簡單的 ResNet 用於梯度分析
    """
    
    def __init__(self, num_blocks, block_type='preact'):
        super().__init__()
        
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        # 堆疊殘差塊
        blocks = []
        for _ in range(num_blocks):
            if block_type == 'preact':
                blocks.append(PreActBasicBlock(64, 64))
            else:
                blocks.append(OriginalBasicBlock(64, 64))
        self.blocks = nn.Sequential(*blocks)
        
        # 最終層
        self.bn_final = nn.BatchNorm2d(64)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, 10)
        
        self.block_type = block_type
    
    def forward(self, x):
        # 初始卷積
        x = F.relu(self.bn1(self.conv1(x)))
        
        # 殘差塊
        x = self.blocks(x)
        
        # 預活化需要最終的 BN-ReLU
        if self.block_type == 'preact':
            x = F.relu(self.bn_final(x))
        
        # 分類頭
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        
        return x

def analyze_gradients(model, num_samples=10):
    """
    分析各層的梯度大小
    """
    model.train()
    
    grad_norms = []
    
    for _ in range(num_samples):
        x = torch.randn(4, 3, 32, 32)
        y = torch.randint(0, 10, (4,))
        
        # 前向傳播
        output = model(x)
        loss = F.cross_entropy(output, y)
        
        # 反向傳播
        model.zero_grad()
        loss.backward()
        
        # 收集梯度
        sample_grads = []
        for name, param in model.named_parameters():
            if param.grad is not None and 'conv' in name:
                sample_grads.append(param.grad.norm().item())
        
        grad_norms.append(sample_grads)
    
    # 平均梯度
    avg_grads = np.mean(grad_norms, axis=0)
    return avg_grads

# 分析不同深度的梯度
num_blocks = 20

model_original = SimpleResNet(num_blocks, 'original')
model_preact = SimpleResNet(num_blocks, 'preact')

grads_original = analyze_gradients(model_original)
grads_preact = analyze_gradients(model_preact)

print(f"梯度分析（{num_blocks} 個殘差塊）：")
print(f"  原始 ResNet 平均梯度: {np.mean(grads_original):.6f}")
print(f"  預活化 ResNet 平均梯度: {np.mean(grads_preact):.6f}")

In [ ]:
# 視覺化梯度流動
fig, ax = plt.subplots(figsize=(14, 6))

layers = range(len(grads_original))
ax.plot(layers, grads_original, 'o-', label='Original ResNet (Post-activation)', 
        color='#F44336', linewidth=2, markersize=6)
ax.plot(layers, grads_preact, 's-', label='Pre-activation ResNet', 
        color='#4CAF50', linewidth=2, markersize=6)

ax.set_xlabel('Layer (Convolution Layer Index)', fontsize=12)
ax.set_ylabel('Gradient Norm', fontsize=12)
ax.set_title('Gradient Flow Comparison', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gradient_flow.png', dpi=150, bbox_inches='tight')
plt.show()

print("圖片已儲存：gradient_flow.png")

## 5. 完整的預活化 ResNet

In [ ]:
class PreActBottleneck(nn.Module):
    """
    預活化 Bottleneck 塊（用於更深的網路）
    
    x → BN → ReLU → 1x1 Conv → BN → ReLU → 3x3 Conv → BN → ReLU → 1x1 Conv → (+x)
    """
    
    expansion = 4
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        width = out_channels // self.expansion
        
        # 預活化 Bottleneck
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(in_channels, width, 1, bias=False)
        
        self.bn2 = nn.BatchNorm2d(width)
        self.conv2 = nn.Conv2d(width, width, 3, stride, 1, bias=False)
        
        self.bn3 = nn.BatchNorm2d(width)
        self.conv3 = nn.Conv2d(width, out_channels, 1, bias=False)
        
        # 捷徑
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 1, stride, bias=False)
    
    def forward(self, x):
        # 預活化
        out = F.relu(self.bn1(x))
        
        # 捷徑
        shortcut = self.shortcut(out)
        
        # 殘差路徑
        out = self.conv1(out)
        out = F.relu(self.bn2(out))
        out = self.conv2(out)
        out = F.relu(self.bn3(out))
        out = self.conv3(out)
        
        # 加法（無後置活化）
        out = out + shortcut
        
        return out

class PreActResNet(nn.Module):
    """
    完整的預活化 ResNet
    """
    
    def __init__(self, block, num_blocks, num_classes=10):
        super().__init__()
        
        self.in_channels = 64
        
        # 初始卷積（標準的後置活化）
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        
        # 殘差層
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        
        # 最終的 BN-ReLU
        self.bn_final = nn.BatchNorm2d(512 * block.expansion)
        
        # 分類頭
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)
        
        # 初始化
        self._init_weights()
    
    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        
        for stride in strides:
            layers.append(block(self.in_channels, out_channels * block.expansion, stride))
            self.in_channels = out_channels * block.expansion
        
        return nn.Sequential(*layers)
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # 初始卷積
        x = self.conv1(x)
        
        # 殘差層
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        # 最終 BN-ReLU
        x = F.relu(self.bn_final(x))
        
        # 分類
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        
        return x

def PreActResNet18():
    return PreActResNet(PreActBasicBlock, [2, 2, 2, 2])

def PreActResNet50():
    return PreActResNet(PreActBottleneck, [3, 4, 6, 3])

# 測試模型
model = PreActResNet18()
x = torch.randn(2, 3, 32, 32)
y = model(x)

print("PreActResNet-18 測試：")
print(f"  輸入形狀: {x.shape}")
print(f"  輸出形狀: {y.shape}")

# 計算參數量
total_params = sum(p.numel() for p in model.parameters())
print(f"  總參數量: {total_params:,}")

## 6. 不同架構變體的比較

In [ ]:
# 視覺化論文中測試的各種架構變體
architectures = [
    {
        'name': 'Original\n(Post-activation)',
        'structure': 'Conv→BN→ReLU→Conv→BN→(+)→ReLU',
        'identity': 'Blocked',
        'error': 6.37,
        'color': '#F44336'
    },
    {
        'name': 'BN after\naddition',
        'structure': 'Conv→BN→ReLU→Conv→BN→(+)→BN→ReLU',
        'identity': 'Blocked',
        'error': 7.93,
        'color': '#FF5722'
    },
    {
        'name': 'ReLU before\naddition',
        'structure': 'BN→ReLU→Conv→BN→ReLU→Conv→ReLU→(+)',
        'identity': 'Clean',
        'error': 8.75,
        'color': '#FF9800'
    },
    {
        'name': 'ReLU-only\npre-activation',
        'structure': 'ReLU→Conv→BN→ReLU→Conv→BN→(+)',
        'identity': 'Clean',
        'error': 6.71,
        'color': '#FFC107'
    },
    {
        'name': 'Full\npre-activation',
        'structure': 'BN→ReLU→Conv→BN→ReLU→Conv→(+)',
        'identity': 'Clean',
        'error': 5.46,
        'color': '#4CAF50'
    }
]

fig, ax = plt.subplots(figsize=(14, 7))

names = [a['name'] for a in architectures]
errors = [a['error'] for a in architectures]
colors = [a['color'] for a in architectures]

bars = ax.bar(names, errors, color=colors, edgecolor='black', linewidth=2)

# 標註數值
for bar, err in zip(bars, errors):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{err}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Error Rate (%)', fontsize=12)
ax.set_title('CIFAR-10 Error Rate: Different ResNet Block Architectures\n(ResNet-110)', fontsize=14)
ax.set_ylim(0, 10)
ax.grid(True, alpha=0.3, axis='y')

# 標記最佳
ax.annotate('BEST!', xy=(4, 5.46), xytext=(4, 3),
            arrowprops=dict(arrowstyle='->', color='green', lw=2),
            fontsize=14, fontweight='bold', color='green', ha='center')

plt.tight_layout()
plt.savefig('architecture_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("圖片已儲存：architecture_comparison.png")
print("\n完全預活化（BN→ReLU→Conv）獲得最佳結果！")

## 7. 深度與表現的關係

In [ ]:
# 論文中的實驗結果
results = {
    'depth': [110, 164, 1001],
    'original': [6.37, 5.93, np.nan],  # 1001 層原始 ResNet 無法收斂
    'preact': [5.46, 5.46, 4.62]
}

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(results['depth']))
width = 0.35

bars1 = ax.bar(x - width/2, results['original'], width, label='Original ResNet', 
               color='#F44336', edgecolor='black')
bars2 = ax.bar(x + width/2, results['preact'], width, label='Pre-activation ResNet', 
               color='#4CAF50', edgecolor='black')

# 標註數值
for bar, val in zip(bars1, results['original']):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val}%', ha='center', va='bottom', fontsize=11)
    else:
        ax.text(bar.get_x() + bar.get_width()/2, 1,
                'Failed to\nconverge', ha='center', va='bottom', fontsize=9, color='red')

for bar, val in zip(bars2, results['preact']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val}%', ha='center', va='bottom', fontsize=11)

ax.set_xlabel('Network Depth', fontsize=12)
ax.set_ylabel('Error Rate (%)', fontsize=12)
ax.set_title('CIFAR-10 Error Rate vs Network Depth', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f'{d} layers' for d in results['depth']])
ax.legend(fontsize=11)
ax.set_ylim(0, 8)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('depth_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("圖片已儲存：depth_comparison.png")
print("\n關鍵發現：")
print("  - 原始 ResNet 無法訓練 1001 層")
print("  - 預活化 ResNet 成功訓練 1001 層，達到 4.62% 錯誤率")

## 總結

### 核心發現

1. **恆等路徑必須乾淨**：後置 ReLU 阻斷恆等映射
2. **預活化優於後置活化**：BN → ReLU → Conv
3. **梯度流動改善**：恆等路徑提供無阻礙的梯度高速公路

### 架構對比

| 特性 | 原始 ResNet | 預活化 ResNet |
|------|------------|---------------|
| 順序 | Conv→BN→ReLU | BN→ReLU→Conv |
| 恆等路徑 | 被 ReLU 阻斷 | 完全乾淨 |
| 最深可訓練 | ~200 層 | 1001+ 層 |

### 實用建議

- 深度網路（> 50 層）：使用預活化
- 淺層網路：兩種都可以
- 第一層保持後置活化
- 記得在分類頭前加 BN-ReLU

In [ ]:
print("="*60)
print("第十五章：深度殘差網路中的恆等映射 實作完成！")
print("="*60)
print("\n本章實作了：")
print("  1. 原始 ResNet 塊（後置活化）")
print("  2. 預活化 ResNet 塊")
print("  3. 恆等映射測試")
print("  4. 梯度流動分析")
print("  5. 完整 PreActResNet")
print("  6. 不同架構變體比較")
print("\n生成的圖片：")
print("  - block_comparison.png：兩種塊的架構圖")
print("  - identity_test.png：恆等映射測試")
print("  - gradient_flow.png：梯度流動比較")
print("  - architecture_comparison.png：不同架構的錯誤率")
print("  - depth_comparison.png：深度與表現關係")